In [1]:
import sys
import os
sys.path.append('patchcore-inspection')

In [2]:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [3]:
import os
import torch
print("Torch imported successfully")

from patchcore.patchcore import PatchCore
print("PatchCore imported successfully")
from torchvision import datasets, transforms
print("Torchvision imported successfully")
from torch.utils.data import DataLoader
print("DataLoader imported successfully")

Torch imported successfully


/Users/vegardhaugerud/.pyenv/versions/3.11.4/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PatchCore imported successfully
Torchvision imported successfully
DataLoader imported successfully


In [11]:
datapath = '../data'  # Update this with the correct path to your dataset
train_dir = os.path.join(datapath, 'leather', 'train')  # leather dataset
test_dir = os.path.join(datapath, 'leather', 'test')
# Image preprocessing and DataLoader
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Initialize PatchCore model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PatchCore(device=device)



In [16]:
def train_patchcore(model, train_loader, device):
    model.train()
    all_features = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            all_labels.append(labels)
    
    all_features = torch.cat(all_features, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    return all_features, all_labels

In [17]:
train_features, train_labels = train_patchcore(model, train_loader, device)


RuntimeError: torch.cat(): expected a non-empty list of Tensors

In [ ]:
import subprocess

def run_leather_evaluation():
    cmd = [
        "python", "../evaluate/evaluate_experiment2.py",
        "--dataset_base_dir", "../data",
        "--anomaly_maps_dir", "../output/anomaly_maps",
        "--output_dir", "../output/metrics",
        "--evaluated_objects", "leather",
        "--pro_integration_limit", "0.3"
    ]
    subprocess.run(cmd, check=True)

if __name__ == "__main__":
    run_leather_evaluation()
    print("Evaluation completed successfully.")